# 01 — Setup & Data Loading

This notebook sets up the environment and loads the ConfAIde benchmark data.
Run cells top to bottom. If you're on Colab, everything should just work.

## Step 1: Install Packages

**What:** Installs all libraries needed for the project.

**Why each one:**
- `transformers` — loads Qwen2.5-7B and runs inference
- `torch` — the underlying deep learning engine
- `sentence-transformers` — embeds NLA descriptions and CI reference phrases for cosine similarity scoring
- `pandas` — stores and filters scenarios as a dataframe
- `numpy` — numerical operations (cosine similarity, aggregations)

**Expected output:** A wall of pip install text, ending with `Successfully installed ...`

**Gotcha:** On Colab, you may need to restart the runtime after installing. If you get import errors below, do Runtime → Restart and run again.

In [ ]:
!pip install -q transformers torch sentence-transformers pandas numpy

## Step 2: Download ConfAIde Benchmark Data

**What:** Fetches the raw benchmark files from the official ConfAIde GitHub repo and saves them to `data/`.

**Why download programmatically:** Keeps the notebook self-contained and always pulls the latest version.

**Expected output:** A confirmation line for each file downloaded, e.g. `✓ Saved data/tier_1.txt`

**Gotcha:** If you're offline or GitHub is down, this will fail. In that case, manually download the files from https://github.com/skywalker023/confaide/tree/main/benchmark and drop them in `data/`.

In [ ]:
import os
import requests

BASE_URL = "https://raw.githubusercontent.com/skywalker023/confaide/main/benchmark/"

FILES = [
    "tier_1.txt",
    "tier_1_labels.txt",
    "tier_2a.txt",
    "tier_2b.txt",
    "tier_2_labels.txt",
    "tier_3.txt",
    "tier_3_control.txt",
    "tier_4.txt",
]

os.makedirs("../data", exist_ok=True)

for filename in FILES:
    response = requests.get(BASE_URL + filename)
    if response.status_code == 200:
        save_path = f"../data/{filename}"
        with open(save_path, "w") as f:
            f.write(response.text)
        print(f"✓ Saved {save_path}")
    else:
        print(f"✗ Failed to download {filename} (status {response.status_code})")

## Step 3: Parse the Benchmark Files

**What:** Loads each tier into a pandas DataFrame with columns `tier` and `scenario`.

**Why two parsers:** The tiers use two different formats:
- **Tiers 1, 2a, 2b** — one scenario per line, plain text
- **Tiers 3, 4** — multi-line dialogue scenarios wrapped in `<BEGIN>` / `<END>` tags

**Expected output:** A summary table showing how many scenarios loaded per tier.

**Gotcha:** Empty lines in the plain-text tiers are filtered out automatically. If a count looks wrong, check the raw file.

In [ ]:
import pandas as pd
import re

def load_flat(filepath, tier_name):
    """Tiers 1, 2a, 2b: one scenario per line."""
    with open(filepath) as f:
        lines = [line.strip() for line in f if line.strip()]
    return pd.DataFrame({"tier": tier_name, "scenario": lines})

def load_dialogue(filepath, tier_name):
    """Tiers 3, 4: multi-line entries wrapped in <BEGIN> ... <END> tags."""
    with open(filepath) as f:
        content = f.read()
    # Extract everything between <BEGIN> and <END>
    scenarios = re.findall(r"<BEGIN>(.*?)<END>", content, re.DOTALL)
    scenarios = [s.strip() for s in scenarios if s.strip()]
    return pd.DataFrame({"tier": tier_name, "scenario": scenarios})

tiers = [
    load_flat("../data/tier_1.txt",  "tier_1"),
    load_flat("../data/tier_2a.txt", "tier_2a"),
    load_flat("../data/tier_2b.txt", "tier_2b"),
    load_dialogue("../data/tier_3.txt", "tier_3"),
    load_dialogue("../data/tier_4.txt", "tier_4"),
]

df = pd.concat(tiers, ignore_index=True)

print("Scenarios loaded per tier:")
print(df.groupby("tier").size().to_string())
print(f"\nTotal: {len(df)} scenarios")

## Step 4: Verify — Sample from Each Tier

**What:** Prints one scenario from each tier so you can sanity-check the data loaded correctly.

**Expected output:** Five scenario snippets, one per tier. Tiers 1/2a/2b will be single-sentence prompts; tiers 3/4 will be multi-line dialogues.

**Gotcha:** If a tier shows an empty string or looks garbled, the parser probably hit a format edge case — let me know and we'll fix the regex.

In [ ]:
for tier_name in ["tier_1", "tier_2a", "tier_2b", "tier_3", "tier_4"]:
    sample = df[df["tier"] == tier_name].iloc[0]["scenario"]
    print(f"{'='*60}")
    print(f"TIER: {tier_name}")
    print(f"{'='*60}")
    # Print first 400 chars to keep output readable
    print(sample[:400] + ("..." if len(sample) > 400 else ""))
    print()